# Практична робота 2. pandas: побудова DataFrame

**Виконав:** Maksym SHmelov · **Група:** IT41 · **Варіант:** 2 — Львів

**Мета:** побудувати `DataFrame`, попрацювати з індексом, `.loc` / `.iloc`, фільтрацією та обчислюваними стовпцями. Дані температур і опадів навчальні, наближені.

## 1. Перевірка середовища та демонстраційний приклад

In [1]:
import numpy as np
import pandas as pd
print("pandas:", pd.__version__, "| NumPy:", np.__version__)

# Демонстраційний магазин з умови
shop = pd.DataFrame({"item":["хліб","молоко","яблука","цукор","чай"],
                     "price":[28,42,55,38,65], "stock":[120,60,90,40,25]})
shop_by_item = shop.set_index("item")
print("Рядок молока через .loc:", shop_by_item.loc["молоко"].to_dict())
shop = shop_by_item.reset_index()
shop.index = range(1, 6)
print("Ціни .loc[1:3]:", shop.loc[1:3, "price"].tolist())
print("Ціни .iloc[0:3, 1]:", shop.iloc[0:3, 1].tolist())
print("Недорогі товари з запасом > 50:", shop.loc[(shop.price < 50) & (shop.stock > 50), "item"].tolist())
shop["total_value"] = shop.price * shop.stock
shop["low_stock"] = shop.stock < 50
def stock_label(qty):
    if qty < 50: return "мало"
    elif qty < 100: return "достатньо"
    return "багато"
shop["stock_label"] = shop.stock.apply(stock_label)
print("Обчислювані стовпці:")
print(shop[["item","total_value","low_stock","stock_label"]].to_string(index=False))

pandas: 2.2.3 | NumPy: 2.3.5
Рядок молока через .loc: {'price': 42, 'stock': 60}
Ціни .loc[1:3]: [28, 42, 55]
Ціни .iloc[0:3, 1]: [28, 42, 55]
Недорогі товари з запасом > 50: ['хліб', 'молоко']
Обчислювані стовпці:
  item  total_value  low_stock stock_label
  хліб         3360      False      багато
молоко         2520      False   достатньо
яблука         4950      False   достатньо
 цукор         1520       True        мало
   чай         1625       True        мало


## 2. DataFrame власного варіанта — Львів

In [2]:
months = ["Січ","Лют","Бер","Кві","Тра","Чер","Лип","Сер","Вер","Жов","Лис","Гру"]
avg_temp = [-3,-2,2,8,13,16,18,17,13,8,3,-1]
precip = [40,38,42,55,85,100,95,78,58,48,52,46]
df = pd.DataFrame({"month":months, "avg_temp":avg_temp, "precip":precip})
print(df.to_string(index=False))
print("\nshape:", df.shape, "\ndtypes:\n", df.dtypes, "\nindex:", df.index)
print("Індекс 0–11:", df.index.equals(pd.RangeIndex(12)))

month  avg_temp  precip
  Січ        -3      40
  Лют        -2      38
  Бер         2      42
  Кві         8      55
  Тра        13      85
  Чер        16     100
  Лип        18      95
  Сер        17      78
  Вер        13      58
  Жов         8      48
  Лис         3      52
  Гру        -1      46

shape: (12, 3) 
dtypes:
 month       object
avg_temp     int64
precip       int64
dtype: object 
index: RangeIndex(start=0, stop=12, step=1)
Індекс 0–11: True


## Завдання 1. Індекс

In [3]:
print("Початковий індекс:", df.index)
by_month = df.set_index("month")
print("\nРядок липня через .loc:")
print(by_month.loc["Лип"])
reset_df = by_month.reset_index()
print("\nПісля reset_index(): індекс", reset_df.index, "| стовпці", reset_df.columns.tolist())

Початковий індекс: RangeIndex(start=0, stop=12, step=1)

Рядок липня через .loc:
avg_temp    18
precip      95
Name: Лип, dtype: int64

Після reset_index(): індекс RangeIndex(start=0, stop=12, step=1) | стовпці ['month', 'avg_temp', 'precip']


## Завдання 2. Відбір рядків і стовпців

In [4]:
df.index = range(1, 13)
print(".loc[1:6, 'avg_temp']:", df.loc[1:6, "avg_temp"].tolist())
print(".iloc[0:6, 1]:", df.iloc[0:6, 1].tolist())
print("month і precip:\n", df[["month", "precip"]].to_string())

# Показуємо, чому loc та iloc можуть обирати різні рядки за індексу 0–11
zero = df.reset_index(drop=True)
print("\nНа індексі 0–11, .loc[1:6]:", zero.loc[1:6, "month"].tolist())
print("На індексі 0–11, .iloc[0:6]:", zero.iloc[0:6]["month"].tolist())

.loc[1:6, 'avg_temp']: [-3, -2, 2, 8, 13, 16]
.iloc[0:6, 1]: [-3, -2, 2, 8, 13, 16]
month і precip:
    month  precip
1    Січ      40
2    Лют      38
3    Бер      42
4    Кві      55
5    Тра      85
6    Чер     100
7    Лип      95
8    Сер      78
9    Вер      58
10   Жов      48
11   Лис      52
12   Гру      46

На індексі 0–11, .loc[1:6]: ['Лют', 'Бер', 'Кві', 'Тра', 'Чер', 'Лип']
На індексі 0–11, .iloc[0:6]: ['Січ', 'Лют', 'Бер', 'Кві', 'Тра', 'Чер']


## Завдання 3. Фільтрація за умовою

In [5]:
print("avg_temp > 15:", df.loc[df.avg_temp > 15, "month"].tolist())
print("avg_temp > 10 і precip > 50:", df.loc[(df.avg_temp > 10) & (df.precip > 50), "month"].tolist())
print("avg_temp < 0 або precip > 80:", df.loc[(df.avg_temp < 0) | (df.precip > 80), "month"].tolist())

avg_temp > 15: ['Чер', 'Лип', 'Сер']
avg_temp > 10 і precip > 50: ['Тра', 'Чер', 'Лип', 'Сер', 'Вер']
avg_temp < 0 або precip > 80: ['Січ', 'Лют', 'Тра', 'Чер', 'Лип', 'Гру']


## Завдання 4. Обчислювані стовпці

In [6]:
# Векторизована формула для температури у Фаренгейтах
df["avg_temp_f"] = df["avg_temp"] * 9 / 5 + 32
# Булева ознака: опадів більше за середнє по таблиці
mean_precip = df["precip"].mean()
df["is_wet"] = df["precip"] > mean_precip
# Визначення сезону за номером місяця за допомогою .apply()
def season_for_month(n):
    if n in (12,1,2): return "зима"
    if n in (3,4,5): return "весна"
    if n in (6,7,8): return "літо"
    return "осінь"
df["season"] = df.index.to_series().apply(season_for_month)
print("Середнє опадів:", round(mean_precip, 2), "мм")
print(df.to_string())
print("\nТипи: avg_temp_f —", df.avg_temp_f.dtype, "; is_wet —", df.is_wet.dtype, "; season —", df.season.dtype)

Середнє опадів: 61.42 мм
   month  avg_temp  precip  avg_temp_f  is_wet season
1    Січ        -3      40        26.6   False   зима
2    Лют        -2      38        28.4   False   зима
3    Бер         2      42        35.6   False  весна
4    Кві         8      55        46.4   False  весна
5    Тра        13      85        55.4    True  весна
6    Чер        16     100        60.8    True   літоо
7    Лип       18      95        64.4    True   літоо
8    Сер        17      78        62.6    True   літо
9    Вер        13      58        55.4   False  осінь
10   Жов         8      48        46.4   False  осінь
11   Лис         3      52        37.4   False  осінь
12   Гру        -1      46        30.2   False   зима

Типи: avg_temp_f — float64 ; is_wet — bool ; season — object


## Письмові відповіді на контрольні питання

**1. Чому `.loc[1:6]` і `.iloc[0:6]` можуть повертати різні рядки?**  
`.loc` вибирає за підписами індексу й включає праву межу діапазону. `.iloc` вибирає за позиціями, причому кінцева позиція не включається. Якщо індекс не збігається з позицією рядка, результати будуть різними. Наприклад, за індексу 0–11 `.loc[1:6]` починає з другого рядка, а `.iloc[0:6]` — з першого.

**2. Чому `and` викликає помилку, а `&` працює?**  
Порівняння стовпців повертає Series із багатьма значеннями True/False. Python-оператор `and` очікує одне загальне булеве значення й не може визначити істинність усієї Series. `&` виконує поелементне логічне «І» між двома Series. Кожну умову потрібно брати в дужки.

**3. Чим відрізняється формула від `.apply()`?**  
Формула створює стовпець векторизованою операцією над усім Series одразу; для стандартної арифметики це зазвичай простіше й швидше. `.apply()` викликає власну функцію для кожного значення (або рядка) і кірисний для складнішої логіки з умовами. У цій роботі формула переводить температуру у Фаренгейти, а `.apply()` визначає сезон.

## Висновок

Побудовано `DataFrame` для варіанта 2 (Львів) із середньомісячними температурами й опадами. Перевірено індекс за замовчуванням, виконано `set_index()` / `reset_index()`, відбір через `.loc` і `.iloc`, вибір стовпців та фільтрацію за кількома умовами. Створено обчислювані стовпці температури у Фаренгейтах, булеву ознаку опадів вище середнього та сезон за допомогою `.apply()`.